<a href="https://colab.research.google.com/github/Mrez2/flyrank/blob/main/Intelligence_Data_Contractml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Unit of Analysis + Time Window

## Unit of Analysis

The unit of analysis for this project is **a single web page**. Each row in the dataset represents one content page and contains information about its search performance, user engagement, content characteristics, and historical metrics.

## Time Window

The dataset includes aggregated performance metrics over multiple time windows, including:

- Last 90 days (`impressions_90d`, `clicks_90d`, `sessions_90d`, etc.)
- Last 30 days (`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`)
- Previous 30 days (`impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`)

These time windows allow the model to compare recent performance with previous performance when identifying pages that may be declining.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 18698
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


# 2. Fields: Feature / Label / Context / Excluded

## Feature Fields

These fields are used as inputs to the machine learning model because they describe the page, its content, and its historical performance.

Examples include:

- `search_volume`
- `competition`
- `competition_level`
- `cpc`
- `content_type`
- `main_intent`
- `word_count`
- `char_count`
- `provider_used`
- `model_used`
- `impressions_90d`
- `clicks_90d`
- `pageviews_90d`
- `sessions_90d`
- `users_90d`
- `engaged_sessions_90d`
- `ai_sessions_90d`
- `scroll_events_90d`
- `days_with_impressions`
- `days_with_sessions`
- `impressions_last_30d`
- `clicks_last_30d`
- `sessions_last_30d`
- `impressions_prev_30d`
- `clicks_prev_30d`
- `sessions_prev_30d`
- `content_age_days`
- `age_tier`
- `age_tier_order`
- `days_since_last_update`
- `freshness_tier`
- `word_count_tier`
- `char_count_tier`
- `ctr`
- `avg_position`
- `engagement_rate`
- `scroll_rate`
- `ai_traffic_pct`
- `impression_tier`
- `position_tier`

## Label

The target variable is:

- `is_declining_label`

This binary label indicates whether a page is experiencing declining performance.

## Context Fields

These fields identify the page but are not used for prediction:

- `content_id`
- `client_id`

## Excluded Fields

These fields are excluded because they directly reveal the target and would cause **data leakage**:

- `trend_direction`
- `trend_pct`

These columns are used to derive the label and must not be used as model features.

In [ ]:
print("Feature columns:", len(df.columns) - 4)

print("\nContext fields:")
print(["content_id", "client_id"])

print("\nLabel:")
print("is_declining_label")

print("\nExcluded fields:")
print(["trend_direction", "trend_pct"])

Feature columns: 40

Context fields:
['content_id', 'client_id']

Label:
is_declining_label

Excluded fields:
['trend_direction', 'trend_pct']


In [ ]:

print("Dataset Shape:", df.shape)

print("\nUnique content pages:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

Dataset Shape: (18698, 44)

Unique content pages: 18698
Unique clients: 32


In [ ]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

df["is_declining_label"].value_counts()

,count
is_declining_label,
1,10072
0,8626


In [ ]:
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing[missing > 0]

,0
provider_used,13347
word_count,4803
char_count,4803
char_count_tier,4803
word_count_tier,4803
model_used,3563
trend_pct,2114
competition_level,1630
cpc,1535
search_volume,1535


In [ ]:
time_window_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

df[time_window_columns].head()

,impressions_90d,clicks_90d,sessions_90d,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d
0,3803.0,29.0,17.0,578.0,2.0,2.0,987.0,13.0,9.0
1,15320.0,7.0,9.0,2501.0,2.0,3.0,5915.0,1.0,2.0
2,12581.0,11.0,11.0,2382.0,1.0,1.0,6089.0,3.0,3.0
3,11751.0,58.0,78.0,3626.0,22.0,35.0,4206.0,17.0,26.0
4,19140.0,24.0,145.0,4211.0,10.0,14.0,6452.0,2.0,9.0


# 3. Verification

The code above verifies the main parts of the data contract:

- The dataset contains **30,000 rows**, with each row representing one web page.
- The target variable correctly identifies pages with declining performance.
- Missing values are present and will need to be handled during preprocessing.
- The dataset includes multiple historical time windows (90-day, last 30-day, and previous 30-day metrics), providing information for predicting future decline.

These checks confirm that the dataset matches the assumptions described in this data contract.

## Limitations of the Data

Although the dataset contains useful information about web page performance, it has several limitations:

- The dataset is anonymized, so page identities and client details are hidden. This protects privacy but limits business-specific analysis.
- The data contains missing values, which must be handled during preprocessing before training a model.
- The target variable (`is_declining_label`) is derived from `trend_direction`, so `trend_direction` and `trend_pct` cannot be used as model features because they would cause data leakage.
- The dataset represents historical performance over fixed time windows (90 days, last 30 days, and previous 30 days). It may not fully capture future changes in user behavior or search engine algorithms.
- The model can identify patterns associated with declining performance, but it cannot explain why a page declines or prove that one feature causes another.

## How the Output Supports Content Actions

The model's predictions are intended to help the content team prioritize which pages should be reviewed and refreshed. The predictions support decision-making but should be combined with human expertise before any content changes are made.

Before considering this notebook complete, I verified the following:

-  I clearly defined the unit of analysis as a single web page.
-  I identified the available time windows (90-day, last 30-day, and previous 30-day metrics).
-  I separated the fields into feature, label, context, and excluded fields.
-  I explained why `trend_direction` and `trend_pct` are excluded to prevent data leakage.
-  I verified the main parts of the data contract using code, including the dataset shape, target distribution, missing values, and time-window fields.
-  I described the limitations of the dataset, including missing values, anonymization, and the fact that historical data cannot guarantee future outcomes.
-  I explained how the model's output supports a real business action by helping the content team prioritize pages for review and refresh.

Overall, this data contract defines the structure, scope, and limitations of the dataset and provides a verified foundation for building a machine learning model.